# Descomissionamento - Teste nas imagens da operação 108_SUB8ag24-108_6000685892 

Autor:  Viviane da Rosa Sommer

Atualizado: 22/05/2025

## Objetivo: Aplicar o modelo model_23-06_Jun no conjunto de imagens da operação 108_SUB8ag24-108_6000685892, para avaliar como o modelo performa nas imagens de descomissionamento no Nordeste.

## Importações necessárias

In [ ]:
# !pip install opencv-python-headless
import os
import shutil
import json
import numpy as np 
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import IoU
from tensorflow.keras.preprocessing import image
from keras.utils import get_custom_objects
import h5py
from utils.predict_functions import jaccard_distance

## Declaração de Constantes e Modelos

In [ ]:
get_custom_objects().update({'jaccard_distance': jaccard_distance})

model_name = 'Modelo/modelcorrigido.h5'
classes_csv = 'Modelo/training_classes.csv'
input_folder = 'Imagens'

## Funções Necessárias

In [ ]:
def fix_name(name):
    return name.replace('/', '_')


def recursive_replace(obj, name_map):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                recursive_replace(v, name_map)
            elif isinstance(v, str) and v in name_map:
                obj[k] = name_map[v]
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            if isinstance(v, (dict, list)):
                recursive_replace(v, name_map)
            elif isinstance(v, str) and v in name_map:
                obj[i] = name_map[v]


def fix_layer_names(model_path, fixed_model_path):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Arquivo '{model_path}' não encontrado.")
    shutil.copy(model_path, fixed_model_path)
    with h5py.File(fixed_model_path, 'r+') as f:
        model_config = f.attrs['model_config']
        if isinstance(model_config, bytes):
            model_config = model_config.decode('utf-8')
        model_config_json = json.loads(model_config)
        name_map = {}
        for layer in model_config_json['config']['layers']:
            old_name = layer['config']['name']
            new_name = fix_name(old_name)
            name_map[old_name] = new_name
            layer['config']['name'] = new_name
            if 'name' in layer:
                layer['name'] = new_name
        recursive_replace(model_config_json, name_map)
        model_weights = f['model_weights']
        for layer_name in list(model_weights.keys()):
            if '/' in layer_name:
                new_name = fix_name(layer_name)
                model_weights.move(layer_name, new_name)

        f.attrs['model_config'] = json.dumps(model_config_json).encode('utf-8')


def load_and_preprocess(img_path, target_size=(512, 512)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0 
    return img_array


def extract_number(filename):
    num = filename.split("/")[-1].split("-")[0]
    return int(num) 

## Carregar modelo

Obs.: rodar a próxima célula apenas na primeira vez. 
Ela corrige o modelo antigo para ser lido pelo Keras 3, gerado o arquivo Modelo/modelcorrigido.h5.

In [ ]:
# fix_layer_names('Modelo/model.h5', model_name)

In [ ]:
model = tf.keras.models.load_model(model_name, custom_objects={
    'loss': BinaryCrossentropy,
    'iou_score': IoU(num_classes=35, target_class_ids=[1] * 35)
})

## Carregar mapeamento das classes do modelo

In [ ]:
with open(classes_csv, 'r') as f:
    class_names = f.readline().strip().split(',')

## Plot dos resultados do modelo

As imagens foram ordenadas de acordo com sua numeração.

As classes foram ordenadas da maior para a menor, em cada imagem

In [ ]:
image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
image_files_sorted = sorted(image_files, key=extract_number)

for img_name in image_files_sorted:
    img_path = os.path.join(input_folder, img_name)
    img_array = load_and_preprocess(img_path)
    input_tensor = np.expand_dims(img_array, axis=0)
    pred = model.predict(input_tensor)[0]

    sorted_indices = np.argsort(pred)[::-1]
    sorted_class_names = [class_names[i] for i in sorted_indices]
    sorted_probs = pred[sorted_indices]

    pred_class = sorted_indices[0]
    pred_prob = sorted_probs[0]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={'width_ratios': [2, 3]})
    ax1.imshow(img_array)
    ax1.axis('off')
    ax1.set_title(f'{img_name}', fontsize=10)

    table_data = [[name, f'{prob:.2f}'] for name, prob in zip(sorted_class_names, sorted_probs)]
    table = ax2.table(cellText=table_data, colLabels=['Classe', 'Prob'], loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.5)
    ax2.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
!jupyter nbconvert --to html Inferência-Imagens.ipynb